# Project 2 | Six-hour wind-speed prediction from ERA5 reanalysis

**EUMETSAT Junior ML Engineer — meteorological data proof-of-concept**

**Goal:** predict wind speed (10 m) **6 hours ahead** at a selected European location, using only information from the prediction issue time and earlier. Benchmark ML against **persistence** (future wind = current wind). Split chronologically so future periods never enter training.

**Critical limitation:** the dataset here is **ERA5 reanalysis**, accessed via Open-Meteo. A model evaluated against later reanalysis is a retrospective ML proof-of-concept, **not an operational weather forecast system**. Reanalysis variables may not be available in real time. Do not make an operational forecast claim.

Source: [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api). Choose a single ERA5 model explicitly and document units and timestamps.

**Start:** Google Colab → New notebook or upload this notebook. Execute cells in order. Data download happens when you run this notebook.

In [ ]:
%pip -q install requests numpy pandas scikit-learn matplotlib joblib

In [ ]:
import json
from pathlib import Path
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib

SEED = 42
LATITUDE, LONGITUDE = 49.8728, 8.6512  # Darmstadt, Germany
START, END = '2022-01-01', '2024-12-31'
HORIZON = 6  # hours
print('Location:', LATITUDE, LONGITUDE, 'Horizon:', HORIZON, 'hours')

## 1. Download hourly reanalysis variables

Use 2022 for training, 2023 for validation, 2024 for held-out testing. All dates are UTC. Wind speed units depend on the API response, so the reported units must be checked.

In [ ]:
url = 'https://archive-api.open-meteo.com/v1/archive'
params = {
    'latitude': LATITUDE, 'longitude': LONGITUDE,
    'start_date': START, 'end_date': END,
    'hourly': 'wind_speed_10m,temperature_2m,relative_humidity_2m,pressure_msl',
    'timezone': 'GMT', 'models': 'era5', 'wind_speed_unit': 'ms',
}
response = requests.get(url, params=params, timeout=120)
response.raise_for_status()
payload = response.json()
if payload.get('error'):
    raise RuntimeError(payload.get('reason', 'Unknown API error'))
print('Units:', payload.get('hourly_units'))
df = pd.DataFrame(payload['hourly'])
df['time'] = pd.to_datetime(df['time'], utc=True)
df = df.set_index('time').sort_index()
print(df.head())
print('Rows:', len(df), 'Missing values:', df.isna().sum().to_dict())

## 2. Target and strictly past/current predictors

At time `t`, the target is measured wind at `t+6h`. Features are known at `t` or earlier only. We use lags of the four available variables, plus time-of-day and annual cycle. For cyclical features, sine/cosine avoid false discontinuity between 23:00 and 00:00.

In [ ]:
features = pd.DataFrame(index=df.index)
variables = ['wind_speed_10m','temperature_2m','relative_humidity_2m','pressure_msl']
for name in variables:
    for lag in [0, 1, 3, 6, 12, 24]:
        features[f'{name}_lag{lag}h'] = df[name].shift(lag)

hour = df.index.hour.to_numpy()
dayofyear = df.index.dayofyear.to_numpy()
features['hour_sin'] = np.sin(2*np.pi*hour/24)
features['hour_cos'] = np.cos(2*np.pi*hour/24)
features['doy_sin'] = np.sin(2*np.pi*dayofyear/365.25)
features['doy_cos'] = np.cos(2*np.pi*dayofyear/365.25)
features['target_wind_ms'] = df['wind_speed_10m'].shift(-HORIZON)
features['persistence_ms'] = df['wind_speed_10m']
ready = features.dropna().copy()
X_cols = [x for x in ready.columns if x not in ['target_wind_ms','persistence_ms']]
# Exclude records whose target time crosses a split-year boundary.
target_time = ready.index + pd.Timedelta(hours=HORIZON)
train = ready[(ready.index.year == 2022) & (target_time.year == 2022)]
valid = ready[(ready.index.year == 2023) & (target_time.year == 2023)]
test = ready[(ready.index.year == 2024) & (target_time.year == 2024)]
print('Train, validation, test:', len(train), len(valid), len(test))
print('Predictors:', len(X_cols))

## 3. Baseline and ML

A model is useful only if it can outperform a reasonable baseline on held-out data. We tune a small set of model configurations using the **2023 validation** set, then evaluate the single selected model on **2024** data.

In [ ]:
def scores(y_true, pred):
    return {
        'MAE_m/s': float(mean_absolute_error(y_true, pred)),
        'RMSE_m/s': float(np.sqrt(mean_squared_error(y_true, pred))),
    }

baseline_valid = scores(valid.target_wind_ms, valid.persistence_ms)
print('Validation persistence:', baseline_valid)

configs = [
    {'max_iter': 100, 'learning_rate': 0.05, 'max_leaf_nodes': 15},
    {'max_iter': 150, 'learning_rate': 0.05, 'max_leaf_nodes': 31},
    {'max_iter': 200, 'learning_rate': 0.03, 'max_leaf_nodes': 31},
]
trials = []
models = []
for cfg in configs:
    candidate = HistGradientBoostingRegressor(random_state=SEED, **cfg)
    candidate.fit(train[X_cols], train.target_wind_ms)
    pred = candidate.predict(valid[X_cols])
    result = scores(valid.target_wind_ms, pred)
    trials.append({'params': cfg, **result})
    models.append(candidate)
print(pd.DataFrame(trials).to_string(index=False))
best_index = int(np.argmin([t['MAE_m/s'] for t in trials]))
best_config = configs[best_index]
print('Chosen configuration:', best_config)

## 4. Final held-out assessment

Train on 2022–2023 (both historical to the 2024 test period). Compare with persistence on 2024 only. Do not modify parameters after inspecting test results.

In [ ]:
train_all = pd.concat([train, valid]).sort_index()
final_model = HistGradientBoostingRegressor(random_state=SEED, **best_config)
final_model.fit(train_all[X_cols], train_all.target_wind_ms)
ml_pred = final_model.predict(test[X_cols])
baseline_pred = test.persistence_ms.to_numpy()
y_test = test.target_wind_ms.to_numpy()
comparison = pd.DataFrame([
    {'model':'Persistence',**scores(y_test, baseline_pred)},
    {'model':'ML gradient boosting',**scores(y_test, ml_pred)},
])
print(comparison.to_string(index=False))
comparison.to_csv('wind_6h_model_metrics.csv',index=False)

sample = 7*24
fig,ax = plt.subplots(figsize=(13,4))
ax.plot(test.index[:sample], y_test[:sample], label='ERA5 verification', linewidth=1.8)
ax.plot(test.index[:sample], baseline_pred[:sample], label='Persistence', alpha=.8)
ax.plot(test.index[:sample], ml_pred[:sample], label='ML prediction', alpha=.8)
ax.set_title('Six-hour-ahead wind speed: first 7 test days (2024)')
ax.set_ylabel('10 m wind speed (m/s)')
ax.legend(); ax.grid(alpha=.2); plt.xticks(rotation=20)
plt.tight_layout(); plt.savefig('wind_6h_prediction.png',dpi=160); plt.show()

joblib.dump({'model':final_model,'features':X_cols,'horizon_hours':HORIZON,
             'location':[LATITUDE,LONGITUDE], 'dataset':'Open-Meteo ERA5 reanalysis',
             'training_years':'2022-2023','test_year':'2024'},'wind_6h_model.joblib')

## 5. Explain and publish the result

- Did ML outperform persistence on **2024 MAE and RMSE**? If not, say so honestly.
- Which weather predictors exist at issue time? We used only current/past inputs.
- Explain what the lag features mean, and why random data splitting would leak future information.
- Why should real operational forecasting use near-real-time observations and/or NWP forecasts rather than later reanalysis?
- In your GitHub README include the API query, timestamps, versions, metrics table, figure, assumptions and limitations.

**Next optional extension:** compare 6-hour and 24-hour horizons *only after the initial model is stable*, with separate baseline and chronological verification.